# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Set device for TensorFlow (use GPU if available, otherwise CPU)
device = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
print(f"Using device: {device}")

Using device: /gpu:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        # First convolutional layer (5x5 kernels, padding='same')
        self.conv1 = tf.keras.layers.Conv2D(channel_1, kernel_size=5, padding='same',
                                           kernel_initializer=initializer)

        # Second convolutional layer (3x3 kernels, padding='same')
        self.conv2 = tf.keras.layers.Conv2D(channel_2, kernel_size=3, padding='same',
                                           kernel_initializer=initializer)

        # Flatten layer
        self.flatten = tf.keras.layers.Flatten()

        # Output fully connected layer
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax',
                                       kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        # First convolutional block: Conv -> ReLU
        x = self.conv1(x)
        x = tf.nn.relu(x)

        # Second convolutional block: Conv -> ReLU
        x = self.conv2(x)
        x = tf.nn.relu(x)

        # Flatten and fully connected output layer
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [10]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every = 200

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=1)

Iteration 0, Epoch 1, Loss: 3.522820472717285, Accuracy: 4.6875, Val Loss: 2.93145751953125, Val Accuracy: 11.5
Iteration 200, Epoch 1, Loss: 2.0749242305755615, Accuracy: 32.14396667480469, Val Loss: 1.8664189577102661, Val Accuracy: 38.80000305175781
Iteration 400, Epoch 1, Loss: 1.929294228553772, Accuracy: 35.97646713256836, Val Loss: 1.7082159519195557, Val Accuracy: 42.29999923706055
Iteration 600, Epoch 1, Loss: 1.855448842048645, Accuracy: 37.908172607421875, Val Loss: 1.694455623626709, Val Accuracy: 41.0


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [12]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, nesterov=True, momentum=0.9)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

print_every = 200
train_part34(model_init_fn, optimizer_init_fn, num_epochs=1)

Iteration 0, Epoch 1, Loss: 2.6836578845977783, Accuracy: 10.9375, Val Loss: 2.933631420135498, Val Accuracy: 11.399999618530273
Iteration 200, Epoch 1, Loss: 1.705569863319397, Accuracy: 40.119712829589844, Val Loss: 1.4787169694900513, Val Accuracy: 50.0
Iteration 400, Epoch 1, Loss: 1.5544322729110718, Accuracy: 45.29301834106445, Val Loss: 1.3320252895355225, Val Accuracy: 53.39999771118164
Iteration 600, Epoch 1, Loss: 1.469497799873352, Accuracy: 48.32310485839844, Val Loss: 1.2371976375579834, Val Accuracy: 56.5


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [13]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 2.5897603034973145, Accuracy: 25.0, Val Loss: 2.808990955352783, Val Accuracy: 11.300000190734863
Iteration 200, Epoch 1, Loss: 2.0916192531585693, Accuracy: 32.38494873046875, Val Loss: 1.818613886833191, Val Accuracy: 40.5
Iteration 400, Epoch 1, Loss: 1.9380728006362915, Accuracy: 35.9375, Val Loss: 1.7311115264892578, Val Accuracy: 42.0
Iteration 600, Epoch 1, Loss: 1.8622431755065918, Accuracy: 37.95497131347656, Val Loss: 1.689814567565918, Val Accuracy: 41.80000305175781


Альтернативный менее гибкий способ обучения:

In [14]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 1.8272 - sparse_categorical_accuracy: 0.3877 - val_loss: 1.6565 - val_sparse_categorical_accuracy: 0.4300
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.6854 - sparse_categorical_accuracy: 0.4170


[1.6853861808776855, 0.4169999957084656]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [15]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)

    layers = [
        tf.keras.layers.Conv2D(32, kernel_size=5, padding='same',
                              kernel_initializer=initializer, input_shape=(32, 32, 3)),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Conv2D(16, kernel_size=3, padding='same',
                              kernel_initializer=initializer),
        tf.keras.layers.Activation('relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax',
                             kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, nesterov=True, momentum=0.9)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

print_every = 200
train_part34(model_init_fn, optimizer_init_fn, num_epochs=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.793527126312256, Accuracy: 6.25, Val Loss: 2.522899866104126, Val Accuracy: 10.300000190734863
Iteration 200, Epoch 1, Loss: 1.8688331842422485, Accuracy: 33.636505126953125, Val Loss: 1.6959443092346191, Val Accuracy: 42.099998474121094
Iteration 400, Epoch 1, Loss: 1.7421185970306396, Accuracy: 38.56764602661133, Val Loss: 1.5970267057418823, Val Accuracy: 46.29999923706055
Iteration 600, Epoch 1, Loss: 1.6733832359313965, Accuracy: 40.89798355102539, Val Loss: 1.5082674026489258, Val Accuracy: 49.599998474121094


In [16]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.5515 - sparse_categorical_accuracy: 0.4546 - val_loss: 1.2864 - val_sparse_categorical_accuracy: 0.5570
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.3161 - sparse_categorical_accuracy: 0.5329


[1.3161300420761108, 0.5328999757766724]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [17]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [18]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.0306897163391113, Accuracy: 4.6875, Val Loss: 3.084655284881592, Val Accuracy: 10.899999618530273
Iteration 200, Epoch 1, Loss: 2.0786890983581543, Accuracy: 32.43936538696289, Val Loss: 1.8475579023361206, Val Accuracy: 38.0
Iteration 400, Epoch 1, Loss: 1.931414246559143, Accuracy: 36.034912109375, Val Loss: 1.7192798852920532, Val Accuracy: 41.10000228881836
Iteration 600, Epoch 1, Loss: 1.8574535846710205, Accuracy: 37.9809684753418, Val Loss: 1.7065436840057373, Val Accuracy: 40.900001525878906


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [19]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        # First convolutional block
        self.conv1 = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same',
                                           kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.dropout1 = tf.keras.layers.Dropout(0.25)

        # Second convolutional block
        self.conv2 = tf.keras.layers.Conv2D(64, kernel_size=3, padding='same',
                                           kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=2)
        self.dropout2 = tf.keras.layers.Dropout(0.25)

        # Third convolutional block
        self.conv3 = tf.keras.layers.Conv2D(128, kernel_size=3, padding='same',
                                           kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.dropout3 = tf.keras.layers.Dropout(0.25)

        # Fourth convolutional block
        self.conv4 = tf.keras.layers.Conv2D(128, kernel_size=3, padding='same',
                                           kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=2)
        self.dropout4 = tf.keras.layers.Dropout(0.25)

        # Flatten and fully connected layers
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(512, activation='relu',
                                        kernel_initializer=initializer)
        self.dropout5 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax',
                                        kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        # First convolutional block: Conv -> BatchNorm -> ReLU -> Dropout
        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout1(x, training=training)

        # Second convolutional block: Conv -> BatchNorm -> ReLU -> MaxPool -> Dropout
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.dropout2(x, training=training)

        # Third convolutional block: Conv -> BatchNorm -> ReLU -> Dropout
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout3(x, training=training)

        # Fourth convolutional block: Conv -> BatchNorm -> ReLU -> MaxPool -> Dropout
        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.dropout4(x, training=training)

        # Flatten and fully connected layers
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout5(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 4.1023664474487305, Accuracy: 7.8125, Val Loss: 23.836963653564453, Val Accuracy: 7.90000057220459
Iteration 700, Epoch 1, Loss: 2.091193199157715, Accuracy: 26.669490814208984, Val Loss: 1.5843749046325684, Val Accuracy: 42.29999923706055
Iteration 1400, Epoch 2, Loss: 1.6254894733428955, Accuracy: 37.06938934326172, Val Loss: 1.3000247478485107, Val Accuracy: 53.20000076293945
Iteration 2100, Epoch 3, Loss: 1.495611310005188, Accuracy: 42.173770904541016, Val Loss: 1.3207125663757324, Val Accuracy: 52.999996185302734
Iteration 2800, Epoch 4, Loss: 1.4047855138778687, Accuracy: 46.123260498046875, Val Loss: 1.1445705890655518, Val Accuracy: 60.39999771118164
Iteration 3500, Epoch 5, Loss: 1.335107684135437, Accuracy: 48.60554885864258, Val Loss: 1.0880950689315796, Val Accuracy: 63.30000305175781
Iteration 4200, Epoch 6, Loss: 1.2922799587249756, Accuracy: 49.98736572265625, Val Loss: 0.9331489205360413, Val Accuracy: 68.19999694824219
Iteration 4900, Epoch

В ходе работы были проведены несколько последовательных экспериментов на CIFAR-10 с использованием разных API Keras и разных архитектур.

Подготовка данных выполнялась на подвыборках размера 49000 (train), 1000 (validation), 10000 (test) с нормализацией по среднему и стандартному отклонению train-выборки. Обучение проводилось с batch_size = 64.

Сначала были протестированы базовые модели: двухслойная полносвязная сеть (hidden_size = 4000, learning_rate = 1e-2, 1 эпоха) и простая трехслойная CNN без дополнительной регуляризации (каналы 32 и 16, learning_rate = 3e-3, SGD + Nesterov momentum = 0.9, 1 эпоха). Эти варианты позволили получить рабочий baseline и проверить корректность пайплайна обучения, однако качество на валидации оставалось ограниченным.

Далее была реализована трехслойная CNN в формате tf.keras.Sequential и обучена двумя способами (кастомный цикл и compile/fit). Для нее использовались параметры Conv(32, k=5) -> Conv(16, k=3) -> Dense(10), learning_rate = 5e-4, SGD + Nesterov momentum = 0.9, 1 эпоха. По динамике метрик результат оказался сопоставимым: качество лучше, чем у полносвязной сети, но модель чувствительна к гиперпараметрам и быстро выходит на плато.

Финальный и наиболее удачный эксперимент выполнен с более глубокой архитектурой CustomConvNet: Conv(64) -> BN -> ReLU -> Dropout(0.25) -> Conv(64) -> BN -> ReLU -> MaxPool -> Dropout(0.25) -> Conv(128) -> BN -> ReLU -> Dropout(0.25) -> Conv(128) -> BN -> ReLU -> MaxPool -> Dropout(0.25) -> Dense(512) -> Dropout(0.5) -> Dense(10, softmax). Обучение: Adam, learning_rate = 1e-3, 10 эпох, print_every = 700. В этом варианте обучение было наиболее стабильным, а регуляризация (BatchNorm + Dropout) снизила переобучение по сравнению с более простыми моделями.

Итоговые выводы:
1. Для CIFAR-10 простых архитектур (особенно чисто полносвязных) недостаточно: они дают baseline, но ограничены по качеству.
2. Увеличение глубины сети (с 2 сверточных слоев до 4) и числа каналов (до 64/128) существенно улучшает качество классификации.
3. Batch Normalization ускоряет и стабилизирует сходимость, а Dropout с вероятностями 0.25 и 0.5 уменьшает переобучение.
4. Комбинация глубокой CNN + Adam + регуляризация является наиболее эффективной среди протестированных вариантов и позволяет уверенно двигаться к требованию по валидационной точности (>= 70% за 10 эпох).